![image.png](https://i.imgur.com/a3uAqnb.png)


# **I-JEPA**
### **Why I-JEPA?**

Normal pixel prediction is very **stochastic**. Due to demanding the model to learn noise, rare extreme cases, and adapt to them or it will be punished by the loss.

for instance:

> an autoencoder not only does it need to reconstruct faithfully the object but also the noise within it and surronding it.

> an image taken in London with a specific camera, if one of the light pixels was `92` instead of `255` due to a dirty camera or weird weather, standard pixel prediction will punish the model for predicting `255`.

### **Feature-Level > Pixel-Level**

* To understand the world, you do not need to understand pixel-level information—you need **feature-level information**
* This is why JEPA doesn't re-create the raw data
* It just learns to create better representations (embeddings) of the data by **masking and predicting the missing patches** (thus; creating embeddings without noise)

### Dataset
we are using the STL-10 dataset, it has 100k unlabled images !
with a small subset of 5k images are actually labled,


![image.png](https://i.imgur.com/a4Gvw08.png)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# STL-10 is 96x96 RGB. These are the standard channel stats for it.
STL_MEAN = (0.4467, 0.4398, 0.4066)
STL_STD  = (0.2603, 0.2566, 0.2713)

IMG_SIZE = 96
BATCH    = 128*2

# JEPA gets its learning signal from masking, so it does NOT need the
# heavy color jitter / blur that contrastive methods (SimCLR, etc.) rely on.
pretrain_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(STL_MEAN, STL_STD),
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(STL_MEAN, STL_STD),
])

# 100k unlabeled images for pretraining the encoder.
pretrain_set = datasets.STL10("./data", split="unlabeled", download=True, transform=pretrain_tf)

# Small labeled sets for evaluation (5k train / 8k test, 10 classes).
train_set = datasets.STL10("./data", split="train", download=True, transform=eval_tf)
test_set  = datasets.STL10("./data", split="test",  download=True, transform=eval_tf)


print(f"pretrain: {len(pretrain_set):>6} images")
print(f"train:    {len(train_set):>6} images")
print(f"test:     {len(test_set):>6} images")

In [ ]:
# --- Data Loaders ---
# Pretraining Loop Loader (Self-Supervised Unlabeled Data)
pretrain_loader = DataLoader(pretrain_set, batch_size=BATCH, shuffle=True, drop_last=True, pin_memory=True)

# Downstream Linear Probe Loaders (Labeled Data for Evaluation)
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH, shuffle=False, pin_memory=True)

### JEPA implementation
**context encoder** and **Target encoder** have identical architectures, except at the context encoder we don't pass all patches, we pass the context (non-masked) patches as input.

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, img_size=96, patch_size=8, in_chans=3, dim=128*2, depth=12, heads=8):
        super().__init__()
        self.patch_size = patch_size
        self.dim = dim
        self.grid = img_size // patch_size

        # Conv2d to patchify and project in one shot
        self.patch_proj = nn.Conv2d(in_chans, dim, kernel_size=patch_size, stride=patch_size)

        # Standard learnable parameters for position embeddings
        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))

        # 3. Use PyTorch's built-in Transformer Encoder instead of manual blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 4,
            activation="gelu",
            batch_first=True,
            norm_first=True # Matching modern SOTA ViT setups
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim, eps=1e-6)

    def forward(self, imgs, context_idx=None):
        # (b, c, w, h) / (B, dim, grid, grid) -> flatten to (B, dim, patches) -> swap to (B, patches, dim)
        tokens = self.patch_proj(imgs).flatten(2).transpose(1, 2)

        # Add positional info *before* we extract the context patches
        tokens = tokens + self.pos_embed

        # If I-JEPA tells us to look only at specific context patches, filter them
        if context_idx is not None:
            # Expand indices to match the feature dimension (B, num_context_patches, dim)
            gather_idx = context_idx.unsqueeze(-1).expand(-1, -1, self.dim)
            tokens = torch.gather(tokens, dim=1, index=gather_idx)

        # Step 4: Run through the standard PyTorch Transformer
        x = self.transformer(tokens)
        return self.norm(x)

### **Predictor Module Flow**
**whole model pipeline:**
- Remember that during training, in each batch we select patches to mask out, since we selected them we have their indices (locations).
- we run the images to the `target encoder` to get all embeddings, we run only the context patches to the `context encoder` to get their embeddings.
- we run the context embeddings with empty embeddings in the `Predictor`, the goal is to use the context embeddings to "fill"/predict the empty embeddings with the actual target embeddings of the masked out patches.

![image.png](https://i.imgur.com/H8yAwHd.png)

Here is the simplified, single-block pipeline of how the data moves through the `Predictor`:

```text
+-------------------------------------------------------------------+
|                     I-JEPA PREDICTOR BLOCK                        |
+-------------------------------------------------------------------+
|                                                                   |
|  Context Tokens  --> [ Linear: in_proj ] --> Add Context Positions|
|                                                     |             |
|  Mask Tokens     ---------------------------> Add Target Positions|
|                                                     |             |
|                                                     v             |
|               [ Concatenate: Context Tokens + Mask Tokens ]       |
|                                                     |             |
|                                                     v             |
|               [ nn.TransformerEncoder (All Transformer Layers)]   |
|                                                     |             |
|                                                     v             |
|                          [ nn.LayerNorm ]                         |
|                                                     |             |
|                                                     v             |
|                    [ Slice Last T (Target Tokens Only) ]          |
|                                                     |             |
|                                                     v             |
|                       [ Linear: out_proj ]                        |
|                                                                   |
+-------------------------------------------------------------------+
                                  |
                                  v
                       [ Target Predictions ]




In [ ]:
import torch
import torch.nn as nn

class Predictor(nn.Module):
    def __init__(self, num_patches, enc_dim=128*2, dim=128, depth=8, heads=8):
        super().__init__()
        self.dim = dim

        self.in_proj = nn.Linear(enc_dim, dim)
        self.out_proj = nn.Linear(dim, enc_dim)

        # Learnable mask token and a learnable position embedding
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed  = nn.Parameter(torch.zeros(num_patches, dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 4,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)

    def forward(self, ctx, ctx_idx, tgt_idx):
        # extract batch size, num of target patches
        B, T = ctx.size(0), tgt_idx.size(1)

        # Project the context tokens (patches embeddings) to the predictor dimension
        ctx_projected = self.in_proj(ctx)

        # Grab the positions of context and target patches
        ctx_pos = self.pos_embed[ctx_idx]                  # [L, dim]
        tgt_pos = self.pos_embed[tgt_idx]                  # [T, dim]

        # Ready context tokens + Add positions to the target mask tokens
        ctx_tokens = ctx_projected + ctx_pos
        mask_tokens = self.mask_token.expand(B, T, -1) + tgt_pos

        # Combine them into one sequence: [Context Tokens ... Mask Tokens]
        x = torch.cat([ctx_tokens, mask_tokens], dim=1)

        x = self.transformer(x)
        x = self.norm(x)

        # Slice out ONLY the target slots (the last T tokens) and project back
        target_predictions = x[:, -T:]
        return self.out_proj(target_predictions)

### **The EMA Momentum Update Formula**

In I-JEPA, the target encoder weights ($\theta_{\text{target}}$) are updated using an **Exponential Moving Average (EMA)** of the context encoder weights ($\theta_{\text{context}}$). This update happens after every training step without using gradients.

$$\theta_{\text{target}} \leftarrow m \cdot \theta_{\text{target}} + (1 - m) \cdot \theta_{\text{context}}$$

Where:
* **$m$**: The **momentum coefficient** (typically a value close to 1, e.g., `0.996` or `0.999`).

following code updates target encoder to get 'closer' to the context encoder

In [ ]:
@torch.no_grad()
def ema_update(target_encoder, context_encoder, m):
    for pt, po in zip(target_encoder.parameters(), context_encoder.parameters()):
        # for each parameter, make it a tiny bit closer to the context encoder
        pt.mul_(m).add_(po, alpha= (1-m) )

### Utility function
a function that carves out random indices for context and target patches,
it's a pretty complicated function for a simple task, make sure you understand its purpose and the input and output to the main function, the rest is just coding knowledge + problem solving and not really Ai knowledge.

In [ ]:
import random, math

RETRIES = 10  # retry budget for finding a context block with enough patches left
def _rect_to_indices(g, top, left, h, w):
    """Helper for sample_ijepa_masks. Flatten a rectangle -> set of patch indices (row*g + col)."""
    return {r * g + c for r in range(top, top + h) for c in range(left, left + w)}


def _sample_block(grid, h, w, rng):
    """Helper for sample_ijepa_masks. Place an h×w block at a random spot; return its patch indices."""
    top = rng.randint(0, grid - h)
    left = rng.randint(0, grid - w)
    return _rect_to_indices(grid, top, left, h, w)


def _block_size(grid, area_frac, aspect):
    """Helper for sample_ijepa_masks. Turn an area fraction + aspect ratio into (h, w) patch dims."""
    a = area_frac * grid * grid
    return (max(1, min(grid, round(math.sqrt(a * aspect)))),
            max(1, min(grid, round(math.sqrt(a / aspect)))))


def _sample_context(grid, h, w, targets, min_ctx, rng):
    """Helper for sample_ijepa_masks. A big block with target patches removed, retried until enough remain."""
    occupied = set().union(*targets)
    context = set()
    for _ in range(RETRIES):
        context = _sample_block(grid, h, w, rng) - occupied
        if len(context) >= min_ctx:
            break
    return sorted(context) if context else [0]


def sample_ijepa_masks(batch_size, grid, n_targets=4, min_ctx=4, rng=None):
    """Sample I-JEPA context/target masks for a batch.

    For each image, hide `n_targets` rectangular target blocks and expose one
    large context block that excludes them. Returns patch indices, batched.

    Returns:
        contexts:     list of `batch_size` lists — visible patch indices per image
                      (all trimmed to the same length L so they stack into a tensor).
        target_lists: list of `n_targets` lists — target[m][b] is the patch indices
                      of the m-th target block for image b.
    """
    rng = rng or random

    # One block size for all targets, one for all contexts (chosen per batch).
    th, tw = _block_size(grid, rng.uniform(0.15, 0.20), rng.uniform(0.75, 1.5))
    ch, cw = _block_size(grid, rng.uniform(0.85, 1.00), 1.0)

    contexts, targets_per_image = [], []
    for _ in range(batch_size):
        targets = [_sample_block(grid, th, tw, rng) for _ in range(n_targets)]
        contexts.append(_sample_context(grid, ch, cw, targets, min_ctx, rng))
        targets_per_image.append(targets)

    # Trim contexts to a shared length so they stack into one batch.
    L = min(len(c) for c in contexts)
    contexts = [sorted(rng.sample(c, L)) for c in contexts]

    # Regroup targets by target-index instead of by image (batching convenience).
    target_lists = [[sorted(targets_per_image[b][m]) for b in range(batch_size)]
                    for m in range(n_targets)]

    return contexts, target_lists

### Training the JEPA model

In [ ]:
# ain't no way device is on cpu might as well train on your phone
device = torch.device("cuda")

# context and target encoders
ctx_enc = Encoder(img_size=96, patch_size=8).to(device)
tgt_enc = Encoder(img_size=96, patch_size=8).to(device)

tgt_enc.load_state_dict(ctx_enc.state_dict())    # why .. ?
tgt_enc.requires_grad_(False)                    # freeze target encoder

# Predictor | num_patches = grid*grid
pred = Predictor(num_patches=144).to(device)

# only context_encoder and predictor should train
params = list(ctx_enc.parameters()) + list(pred.parameters())
opt = torch.optim.AdamW(params, lr=5e-5, weight_decay=0.001)

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def train(epochs=8, ema_start=0.996, ema_end=0.9999, device=device, loader=pretrain_loader, opt=opt):
    total = epochs * len(loader)
    rng = random.Random(0)
    step = 0

    step_losses  = []    # loss at gradient update (step)
    epoch_losses = []    # mean loss per epoch

    for epoch in range(epochs):
        running = 0.0
        pbar = tqdm(loader, desc=f"epoch {epoch+1}/{epochs}")
        for imgs, _ in pbar:
            imgs = imgs.to(device)

            # Sample context + target masks for this batch.
            cl, tls = sample_ijepa_masks(imgs.size(0), ctx_enc.grid, rng=rng)
            ci  = torch.tensor(cl, device=device)
            tis = [torch.tensor(t, device=device) for t in tls]

            # normalize target outputs
            with torch.no_grad():
                full = F.layer_norm(tgt_enc(imgs), (ctx_enc.dim,))

            # encodes context; predictor reconstructs each target block.
            ctx_emb = ctx_enc(imgs, ci)
            total_loss = 0.0
            for ti in tis:
                guess  = pred(ctx_emb, ci, ti)
                # grab actual target embeddings
                answer = full.gather(1, ti.unsqueeze(-1).expand(-1, -1, ctx_enc.dim))
                total_loss += F.mse_loss(guess, answer)
            loss = total_loss / len(tis)

            opt.zero_grad()
            loss.backward()
            opt.step()

            # Update target encoder (EMA of context); m ramps 0.996 -> ema_end
            m = ema_start + (ema_end - ema_start) * (step / max(1, total - 1))
            ema_update(tgt_enc, ctx_enc, m)

            # monitoring
            step_losses.append(loss.item())
            running += loss.item()
            step += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        epoch_losses.append(running / len(loader))
        pbar.set_postfix(loss=f"{epoch_losses[-1]:.4f}")
    return step_losses, epoch_losses

In [ ]:
step_losses, epoch_losses = train(epochs=60, device=device)

In [ ]:
first, last = epoch_losses[0], epoch_losses[-1]
drop = (first - last) / first * 100
print(f"epoch 1 loss: {first:.4f}")
print(f"final  loss: {last:.4f}")
print(f"reduced by {drop:.1f}%")

# --- plot ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(step_losses, alpha=0.4)
ax[0].set_title("per-step loss (noisy)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("smooth L1 loss")

ax[1].plot(range(1, len(epoch_losses) + 1), epoch_losses, marker="o")
ax[1].set_title("per-epoch mean loss (trend)")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("smooth L1 loss")

plt.tight_layout()
plt.show()

### **Linear Probing**

To evaluate the quality of our model's self-supervised embeddings, we will use a standard technique called **linear probing**.

* **The Setup:** We have 5,000 labeled images. We will pass them through our frozen I-JEPA encoder to generate rich feature embeddings.
* **The Goal:** If these learned features are truly high quality, they should make the data **linearly separable** across different classes.
* **The Test:** A single, vanilla linear layer (`nn.Linear`) with zero non-linearities (like ReLU) should be sufficient to classify the data using a basic linear transformation:

$$y = Wx + b$$

> **Why this matters:** We aren't fine-tuning the encoder here. we're just checking if we can draw lines in the encoded dimensions to seperate the classes.

In [ ]:
import torch
from tqdm import tqdm

@torch.no_grad()
def extract_features(encoder, data_loader, device):
    """Passes images through the frozen encoder and flattens the patch grid.

    Shape transition: [B, N, D] -> [B, N * D]
    """
    encoder.eval()
    feats, labels = [], []

    for imgs, y in tqdm(data_loader, desc="Extracting features"):
        # Forward pass through encoder -> Shape: [B, N, D]
        p = encoder(imgs.to(device)).float()

        # Flatten patches (N) and dimensions (D) into a single vector -> Shape: [B, N * D]
        flat_features = p.flatten(1)

        feats.append(flat_features.cpu())
        labels.append(y)

    return torch.cat(feats), torch.cat(labels)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

def linear_probe(feature_extractor, extract_features, train_loader, test_loader, device,
                 num_classes=10, epochs=30, lr=1e-3):
    # Freeze encoder, extract features ONCE (no need to re-encode).
    feature_extractor.requires_grad_(False)
    Xtr, ytr = extract_features(feature_extractor, train_loader, device)
    Xte, yte = extract_features(feature_extractor, test_loader,  device)

    # (optional - better results) standardize features.
    mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True) + 1e-6
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

    # move to device
    Xtr, ytr, Xte, yte = Xtr.to(device), ytr.to(device), Xte.to(device), yte.to(device)

    # one linear Wx+b; features are linearly seprable
    clf = nn.Linear(Xtr.size(1), num_classes).to(device)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-4)

    # Train classifier on the features
    accs = []
    for epoch in range(epochs):
        clf.train()
        # manual batching, but why...? (you can replace this with normal batching btw)
        perm = torch.randperm(Xtr.size(0), device=device)
        for i in range(0, len(perm), 256):
            idx = perm[i:i+256]
            loss = F.cross_entropy(clf(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()

        # Test accuracy this epoch.
        clf.eval()
        with torch.no_grad():
            acc = (clf(Xte).argmax(1) == yte).float().mean().item()
        accs.append(acc)

    print(f"linear-probe test accuracy: {max(accs)*100:.2f}%  (best over {epochs} epochs)")
    return clf, accs

In [ ]:
# freeze extracter
clf, probe_accs = linear_probe(ctx_enc, extract_features, train_loader, test_loader, device)

### Checking features generated by AE (SSL with AE)

##### Autoencoders are well known models in self-supervised learning
##### and they produce great embeddings for reconstruction tasks


![image.png](https://i.imgur.com/LrrUL96.png)



The idea is to train the whole architecture:

Input = Image

- Image => Encoder (Compression)

- Compressed representation resides in => Latent space (where we get embeddings)

- Decoder (Decompression) => Reconstructed image

optimize the loss $(Image - Reconstructed)^2$

**Result?**

> The encoder efficiently and accurately compresses images == well seperated representations for different objects.

> one downside is the embeddings contain noise for accurate pixel reconstruction

> Embedding space that is structured for many tasks such as: Retrieval/Data Compression/Transfer Learning and more...


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

def conv_bn(in_c, out_c):
    """one Conv block (stride 2, halves H/W) -> BatchNorm -> ReLU."""
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=4, stride=2, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

def deconv_bn(in_c, out_c, last=False):
    """one ConvTranspose block (stride 2, doubles H/W) -> BatchNorm -> ReLU (or Tanh on last)."""
    layers = [nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1)]
    if last:
        layers += [nn.Tanh()]              # output back to image range
    else:
        layers += [nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)]
    return nn.Sequential(*layers)


class ConvEncoder(nn.Module):
    def __init__(self, in_chans=3, dim=256):
        super().__init__()
        self.net = nn.Sequential(
            conv_bn(in_chans, 32),   # 96 -> 48
            conv_bn(32, 64),         # 48 -> 24
            conv_bn(64, 128),        # 24 -> 12
            conv_bn(128, dim),       # 12 -> 6
        )
        self.dim = dim

    def forward(self, x):
        return self.net(x)           # [B, dim, 6, 6]

    def embed(self, x):
        """One feature vector per image: mean-pool the bottleneck map."""
        return self.forward(x).mean(dim=(2, 3))   # [B, dim]


class ConvDecoder(nn.Module):
    def __init__(self, out_chans=3, dim=256):
        super().__init__()
        self.net = nn.Sequential(
            deconv_bn(dim, 128),          # 6 -> 12
            deconv_bn(128, 64),           # 12 -> 24
            deconv_bn(64, 32),            # 24 -> 48
            deconv_bn(32, out_chans, last=True),  # 48 -> 96
        )

    def forward(self, z):
        return self.net(z)


class Autoencoder(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.encoder = ConvEncoder(dim=dim)
        self.decoder = ConvDecoder(dim=dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
def train_autoencoder(ae, loader, device, epochs=15, lr=1e-4):
    ae.to(device)
    opt = torch.optim.AdamW(ae.parameters(), lr=lr)
    epoch_losses = []

    for epoch in range(epochs):
        ae.train()
        running = 0.0
        pbar = tqdm(loader, desc=f"AE epoch {epoch+1}/{epochs}")
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            # corrupt the INPUT (denoising AE)
            noisy = imgs + 0.3 * torch.randn_like(imgs)

            recon = ae(imgs)
            loss = F.mse_loss(recon, imgs)     # reconstruct the input

            opt.zero_grad()
            loss.backward()
            opt.step()

            running += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        epoch_losses.append(running / len(loader))

    return epoch_losses


ae = Autoencoder(dim=256)
ae_losses = train_autoencoder(ae, pretrain_loader, device, epochs=30)

In [ ]:
@torch.no_grad()
def extract_features_ae(encoder, data_loader, device):
    encoder.eval()
    feats, labels = [], []
    for imgs, y in tqdm(data_loader, desc="extracting features"):
        feats.append(encoder.embed(imgs.to(device)).cpu())   # [B, dim]
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)
import types


In [ ]:
ae_clf, ae_accs = linear_probe(ae.encoder, extract_features_ae,
                               train_loader, test_loader, device)

### Congrats !
now you have a cool feature extractor, you can use this for image search, object detection, transfer learning ...  
lab cooked by me (Muhannad Alhumaidi)

![image.png](https://i.imgur.com/CNQQS0Y.jpeg)
